# Week 4: BERT Fine-Tuning for Fake News Detection

This notebook fine-tunes `distilbert-base-uncased` on the TruthLens fake news dataset. It demonstrates dataset preparation, tokenizer integration, model training, evaluation, and artifact saving.

## 1. Setup and imports

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score
from sklearn.model_selection import train_test_split
from transformers import (DistilBertForSequenceClassification, DistilBertTokenizerFast, TrainingArguments, Trainer, DataCollatorWithPadding, EarlyStoppingCallback)

PROJECT_ROOT = None
for candidate in [Path.cwd()] + list(Path.cwd().parents):
    if (candidate / 'backend').exists() and (candidate / 'data').exists():
        PROJECT_ROOT = candidate
        break
if PROJECT_ROOT is None:
    raise RuntimeError('Could not locate project root containing backend/ and data/')

sys.path.insert(0, str(PROJECT_ROOT))

from backend.preprocess import clean_text, download_nltk_resources

DATA_DIR = PROJECT_ROOT / 'data'
MODEL_DIR = PROJECT_ROOT / 'backend' / 'models'
MODEL_DIR.mkdir(parents=True, exist_ok=True)

print('Project root:', PROJECT_ROOT)
print('Torch device:', torch.device('cuda' if torch.cuda.is_available() else 'cpu'))

d:\Truthlens\.venv\Lib\site-packages\transformers\utils\generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
d:\Truthlens\.venv\Lib\site-packages\transformers\utils\generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
d:\Truthlens\.venv\Lib\site-packages\transformers\utils\generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(


Project root: d:\Truthlens
Torch device: cuda


## 2. Download NLP resources and load data

In [2]:
download_nltk_resources()

real_df = pd.read_csv(DATA_DIR / 'True.csv')
fake_df = pd.read_csv(DATA_DIR / 'Fake.csv')
real_df['label'] = 0
fake_df['label'] = 1
df = pd.concat([real_df, fake_df], ignore_index=True)
df = df[['title', 'text', 'label']].fillna('')

print('Total rows:', len(df))
print(df['label'].value_counts())

Total rows: 44898
label
1    23481
0    21417
Name: count, dtype: int64


## 3. Preprocess text and tokenize

In [3]:
df['combined_text'] = (df['title'] + ' ' + df['text']).astype(str)
df['cleaned_text'] = df['combined_text'].apply(clean_text)

train_df, test_df = train_test_split(df, test_size=0.20, stratify=df['label'], random_state=42)
train_df, val_df = train_test_split(train_df, test_size=0.125, stratify=train_df['label'], random_state=42)

print('Train size:', len(train_df))
print('Validation size:', len(val_df))
print('Test size:', len(test_df))

Train size: 31428
Validation size: 4490
Test size: 8980


In [4]:
tokenizer = DistilBertTokenizerFast.from_pretrained('distilbert-base-uncased')

def encode_texts(df):
    enc = tokenizer(df['cleaned_text'].tolist(), truncation=True, padding=False, max_length=256)
    enc['labels'] = df['label'].tolist()
    return enc

train_enc = encode_texts(train_df)
val_enc = encode_texts(val_df)
test_enc = encode_texts(test_df)

class NewsDataset(torch.utils.data.Dataset):
    def __init__(self, encodings):
        self.encodings = encodings
    def __len__(self):
        return len(self.encodings['input_ids'])
    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
        return item

train_dataset = NewsDataset(train_enc)
val_dataset = NewsDataset(val_enc)
test_dataset = NewsDataset(test_enc)

print('Train dataset size:', len(train_dataset))

Train dataset size: 31428


## 4. Create the DistilBERT model

In [5]:
model = DistilBertForSequenceClassification.from_pretrained('distilbert-base-uncased', num_labels=2)
model.to(torch.device('cuda' if torch.cuda.is_available() else 'cpu'))

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

training_args = TrainingArguments(
    output_dir=str(MODEL_DIR / 'distilbert-finetuned'),
    evaluation_strategy='epoch',
    save_strategy='epoch',
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    logging_dir=str(PROJECT_ROOT / 'logs' / 'bert_training'),
    logging_steps=100,
    report_to='none',
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {
        'accuracy': (predictions == labels).astype(float).mean(),
        'precision': precision_score(labels, predictions),
        'recall': recall_score(labels, predictions),
        'f1': f1_score(labels, predictions),
    }

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.weight', 'classifier.bias', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


## 5. Train the model

In [6]:
trainer.train()

  0%|          | 0/5895 [00:00<?, ?it/s]

You're using a DistilBertTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


{'loss': 0.1949, 'learning_rate': 1.96607294317218e-05, 'epoch': 0.05}
{'loss': 0.0241, 'learning_rate': 1.9321458863443597e-05, 'epoch': 0.1}
{'loss': 0.0174, 'learning_rate': 1.8982188295165395e-05, 'epoch': 0.15}
{'loss': 0.005, 'learning_rate': 1.8642917726887192e-05, 'epoch': 0.2}
{'loss': 0.0184, 'learning_rate': 1.8303647158608994e-05, 'epoch': 0.25}
{'loss': 0.0091, 'learning_rate': 1.796437659033079e-05, 'epoch': 0.31}
{'loss': 0.0133, 'learning_rate': 1.762510602205259e-05, 'epoch': 0.36}
{'loss': 0.0138, 'learning_rate': 1.7285835453774387e-05, 'epoch': 0.41}
{'loss': 0.0078, 'learning_rate': 1.6946564885496184e-05, 'epoch': 0.46}
{'loss': 0.0142, 'learning_rate': 1.6607294317217982e-05, 'epoch': 0.51}
{'loss': 0.0175, 'learning_rate': 1.626802374893978e-05, 'epoch': 0.56}
{'loss': 0.0046, 'learning_rate': 1.5928753180661577e-05, 'epoch': 0.61}
{'loss': 0.0088, 'learning_rate': 1.558948261238338e-05, 'epoch': 0.66}
{'loss': 0.0105, 'learning_rate': 1.5250212044105175e-05, 'e

  0%|          | 0/141 [00:00<?, ?it/s]

{'eval_loss': 0.014712689444422722, 'eval_accuracy': 0.9968819599109131, 'eval_precision': 0.9982920580700256, 'eval_recall': 0.995741056218058, 'eval_f1': 0.9970149253731344, 'eval_runtime': 43.1777, 'eval_samples_per_second': 103.989, 'eval_steps_per_second': 3.266, 'epoch': 1.0}
{'loss': 0.0116, 'learning_rate': 1.3214588634435962e-05, 'epoch': 1.02}
{'loss': 0.014, 'learning_rate': 1.2875318066157762e-05, 'epoch': 1.07}
{'loss': 0.0001, 'learning_rate': 1.253604749787956e-05, 'epoch': 1.12}
{'loss': 0.0049, 'learning_rate': 1.2196776929601357e-05, 'epoch': 1.17}
{'loss': 0.0028, 'learning_rate': 1.1857506361323157e-05, 'epoch': 1.22}
{'loss': 0.0001, 'learning_rate': 1.1518235793044954e-05, 'epoch': 1.27}
{'loss': 0.0066, 'learning_rate': 1.1178965224766754e-05, 'epoch': 1.32}
{'loss': 0.0001, 'learning_rate': 1.0839694656488552e-05, 'epoch': 1.37}
{'loss': 0.0074, 'learning_rate': 1.0500424088210348e-05, 'epoch': 1.42}
{'loss': 0.0015, 'learning_rate': 1.0161153519932147e-05, 'epo

  0%|          | 0/141 [00:00<?, ?it/s]

{'eval_loss': 0.014886479824781418, 'eval_accuracy': 0.998218262806236, 'eval_precision': 0.9982964224872232, 'eval_recall': 0.9982964224872232, 'eval_f1': 0.9982964224872232, 'eval_runtime': 42.5762, 'eval_samples_per_second': 105.458, 'eval_steps_per_second': 3.312, 'epoch': 2.0}
{'loss': 0.0044, 'learning_rate': 6.429177268871926e-06, 'epoch': 2.04}
{'loss': 0.0028, 'learning_rate': 6.0899067005937244e-06, 'epoch': 2.09}
{'loss': 0.0001, 'learning_rate': 5.750636132315522e-06, 'epoch': 2.14}
{'loss': 0.0001, 'learning_rate': 5.41136556403732e-06, 'epoch': 2.19}
{'loss': 0.0001, 'learning_rate': 5.072094995759118e-06, 'epoch': 2.24}
{'loss': 0.0, 'learning_rate': 4.732824427480917e-06, 'epoch': 2.29}
{'loss': 0.0007, 'learning_rate': 4.393553859202715e-06, 'epoch': 2.34}
{'loss': 0.0, 'learning_rate': 4.054283290924513e-06, 'epoch': 2.39}
{'loss': 0.0, 'learning_rate': 3.7150127226463105e-06, 'epoch': 2.44}
{'loss': 0.0, 'learning_rate': 3.375742154368109e-06, 'epoch': 2.49}
{'loss':

  0%|          | 0/141 [00:00<?, ?it/s]

{'eval_loss': 0.011954308487474918, 'eval_accuracy': 0.9984409799554566, 'eval_precision': 0.998297147722435, 'eval_recall': 0.9987223168654173, 'eval_f1': 0.9985096870342771, 'eval_runtime': 42.6612, 'eval_samples_per_second': 105.248, 'eval_steps_per_second': 3.305, 'epoch': 3.0}
{'train_runtime': 2878.3162, 'train_samples_per_second': 32.757, 'train_steps_per_second': 2.048, 'train_loss': 0.008997950469561559, 'epoch': 3.0}


TrainOutput(global_step=5895, training_loss=0.008997950469561559, metrics={'train_runtime': 2878.3162, 'train_samples_per_second': 32.757, 'train_steps_per_second': 2.048, 'train_loss': 0.008997950469561559, 'epoch': 3.0})

## 6. Evaluate on the holdout test set

In [7]:
metrics = trainer.evaluate(test_dataset)
print(metrics)

model.save_pretrained(MODEL_DIR / 'distilbert-finetuned')
tokenizer.save_pretrained(MODEL_DIR / 'distilbert-finetuned')
print('Saved fine-tuned DistilBERT model to', MODEL_DIR / 'distilbert-finetuned')

  0%|          | 0/281 [00:00<?, ?it/s]

{'eval_loss': 0.0075393542647361755, 'eval_accuracy': 0.9992204899777283, 'eval_precision': 0.9991483925910155, 'eval_recall': 0.9993611584327087, 'eval_f1': 0.9992547641860959, 'eval_runtime': 85.2406, 'eval_samples_per_second': 105.349, 'eval_steps_per_second': 3.297, 'epoch': 3.0}
Saved fine-tuned DistilBERT model to d:\Truthlens\backend\models\distilbert-finetuned


## 7. Notes and next steps

- Use the saved model and tokenizer in the FastAPI backend.
- The next milestone is building `backend/main.py` for inference endpoints.
- Optionally, add a small React UI for article input and prediction display.